# Analysis 21 — which topics come up together?

Which topics are discussed in the same conversation? (log: `analysis-log.md`, Analysis 21).

Choices so far (student): topics are found with keyword lists; a conversation ends after 12 hours without any message (as in notebook 16); one line per pair of topics in the network graph.

In [ ]:
import re

import pandas as pd

from wa_analyzer.data import load_own_chat

GAP_HOURS = 12  # silence that ends a conversation (student, same as notebook 16)
GLUED_SYSTEM_LINE = re.compile(
    r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - "
)  # parser glitch; carries real names
NOT_A_TEXT = re.compile(
    r"<Media weggelaten>|<Video note omitted>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand"
)

own = load_own_chat().sort_values("timestamp").reset_index(drop=True)
own["conversation_id"] = (
    own["timestamp"].diff() > pd.Timedelta(hours=GAP_HOURS)
).cumsum()
text = own["message"].fillna("").astype(str)
is_text = ~text.str.contains(GLUED_SYSTEM_LINE) & ~text.str.contains(NOT_A_TEXT)

print(f"Messages: {len(own):,}; conversations: {own['conversation_id'].nunique():,}")

## Topics and their keyword lists

One regular expression per topic. Words are the student's (2026-11-03); the assistant only added
spellings of those words found in the chat (plurals, verb forms). `\b` keeps a keyword from matching
inside a longer word. Words marked `# check` can mean something else; the hand check decides.

In [ ]:
def words(*keywords: str) -> str:
    """One case-insensitive pattern that matches any of the keywords as whole words."""
    return r"(?i)\b(?:" + "|".join(keywords) + r")\b"


# Planning: notebook 16's pattern without the drink words (student: drinks are their own topic)
WHO = r"(?i)\b(?:wie|iemand)\b"
ACTIVITY = r"(?i)\b(?:vnv|afspreken|vanavond|weekend|wat\s+doen|iets\s+doen)\b"
EXTRA = (
    r"(?i)\b(?:(?:maan|dins|woens|donder|vrij|zater|zon)dag(?:avond|middag|ochtend)?|morgen(?:avond|middag)?|straks"
    r"|komen|komt|mee|samen)\b"
)
WHERE = r"(?i)\bwaar\s+(?:doen|zullen|gaan|willen)\s+we\b"

# Football: notebook 14's list (clubs, terms, players, countries, nicknames) and scores like "2-0"
FOOTBALL = words(
    "fc twente",
    "ajax",
    "psv",
    "feyenoord",
    "heracles",
    "psg",
    "manchester city",
    "arsenal",
    "manchester united",
    "tottenham",
    "spurs",
    "juventus",
    "fc barcelona",
    "real madrid",
    "voetbal",
    "wedstrijd",
    "football",
    "goals?",
    "kaart",
    "overtreding",
    "oranje",
    "ek",
    "wk",
    "eredivisie",
    "champions league",
    "penalty",
    "scheids",
    "scheidsrechter",
    "doelpunt",
    "elftal",
    "buitenspel",
    "var",
    "spits",
    "topspits",
    "keeper",
    "wereldkampioen",
    "qatar",
    "scoren",
    "scoort",
    "scoorde",
    "gescoord",
    "ronaldo",
    "messi",
    "mbapp[eé]",
    "neymar",
    "haaland",
    "depay",
    "gakpo",
    "salah",
    "kane",
    "lewandowski",
    "modri[cć]",
    "de bruyne",
    "bellingham",
    "yamal",
    "schouten",
    "hooiveld",
    "weghorst",
    "bergwijn",
    "van persie",
    "nederland",
    "frankrijk",
    "argentini[eë]",
    "engeland",
    "engelse",
    "spanje",
    "senegal",
    "wout",
    "woutje",
)
SCORE = r"(?<![\d:/-])\b\d-\d\b(?![\d:/-])"

TOPICS: dict[str, str] = {
    "football": FOOTBALL + "|" + SCORE,
    "having a drink": words(
        "bier", "biertjes?", "pils(?:je|jes)?", "drankjes?", "potten", "drinken"
    ),
    "laughter": (  # haha variants, lol, laughing emoji, or a message of only emoji
        r"(?i)\b(?:ha){2,}h?\b|\bhah\b|\blol\b|😂|🤣|^(?=.*[^\x00-\x7f])[\W_]+$"
    ),
    "birthdays": words(
        "van harte",
        "gefeliciteerd",
        "verjaardag(?:en)?",
        "jarige?",  # added after hand check 1 (assistant's suggestion, student agreed)
    ),
    "bad news": words(
        "sterkte",
        "vreselijke?",
        "wat naar",
        "verschrikkelijke?",
        "beterschap",  # added after hand check 1
    ),
    "food": words(
        "chappen",
        "eten",
        "smaak",
        "smaakt",
        "restaurants?",
        "afhalen",
        "pizza",
        "patat",
        "bbq",
        "barbecue",  # added after hand check 1
        # "lekker" dropped after hand check 1: 7 of 15 food matches were "well done" / "alright man"
    ),
    "work": words("baan", "werk", "collega", "kantoor", "werken", "vroeg opstaan"),
    "money": words("betalen", "betaald", "geld", "tikkie", "betaalverzoek", "kosten"),
    "holiday": words(
        "vakantie",
        "vlucht",
        "zon",  # check: "zon" is also weather
        "skiën",
        "skieen",
        "hotel",
        "cura[cç]ao",
        "sri lanka",
        # duitsland, italië, spanje dropped (student): countries are also football talk (tournaments)
        "vliegen",
        "vliegtuig",
        "vliegveld",  # added after hand check 1
    ),
    "festive days": words(
        "verkleden",
        "verkleed",
        "outfits?",
        "kerst",
        "optocht",
        "oud en nieuw",
        "carnavals?",
        "koningsdag",  # added after hand check 1
    ),
    "politics": words(
        "verkiezingen",
        "verkiezing",
        "trump",
        "rutte",
        "jetten",
        "tweede kamer",
        "stemmen",  # check: also voting in a WhatsApp poll
        "uitslag",  # check: also a match result
        "pvv",
        "vvd",
        "groenlinks",
        "pvda",
        "democraten",
        "republikeinen",
    ),
    "family": words(
        "kind",
        "kinderen",
        "zus",
        "zusje",
        "broer",  # check: "broer" is also slang for "mate"
        "broertje",
        "pap",
        "papa",
        "mam",
        "mama",
        "zwanger",
        "opa",
        "oma",
        "bruiloft",
        "neef",
        "neefje",
        "nicht",
        "nichtje",
    ),
}

## How often each topic comes up

Messages and conversations that mention each topic.

In [ ]:
own["planning"] = is_text & (
    (text.str.contains(WHO) & (text.str.contains(ACTIVITY) | text.str.contains(EXTRA)))
    | text.str.contains(WHERE)
)
for topic, pattern in TOPICS.items():
    own[topic] = is_text & text.str.contains(pattern)
# Payment requests count for money only (student, hand checks 1 and 2): their fixed text matched football
# ("nederland": 29 of 30), and what they are for ("Patat carnaval") matched food and holidays.
is_payment_request = text.str.contains(r"(?i)betaalverzoek|payreq|tikkie\.me")
for topic in ["planning", *TOPICS]:
    if topic != "money":
        own[topic] &= ~is_payment_request
print(f"Payment requests (money only): {is_payment_request.sum()}")

topic_columns = ["planning", *TOPICS]
per_conversation = own.groupby("conversation_id")[topic_columns].any()

pd.DataFrame(
    {
        "messages": own[topic_columns].sum(),
        "conversations": per_conversation.sum(),
        "share of conversations": per_conversation.mean().round(3),
    }
).sort_values("conversations", ascending=False)

## Hand check: are the matches really about the topic?

A sample of matched messages per topic, labelled by hand (student) in `is_topic_label` (y/n). More messages for topics with words that can mean something else. Football was checked in notebook 14. The file is not overwritten once it exists, so the labels stay.

In [ ]:
from wa_analyzer.data import PROCESSED

SEED = 42
LABEL_FILE = PROCESSED / "topic-label-check.csv"
SAMPLE_SIZE = {
    "food": 15,
    "holiday": 15,
    "politics": 15,
    "family": 15,
}  # topics with words to check
DEFAULT_SAMPLE = 10

none = (~per_conversation.any(axis=1)).sum()
print(f"Conversations without any topic: {none:,} of {len(per_conversation):,}")
print(
    f"Topics per conversation: {per_conversation.sum(axis=1).value_counts().sort_index().to_dict()}"
)

if LABEL_FILE.exists():
    print(f"{LABEL_FILE.name} exists, not overwritten")
else:
    samples = pd.concat(
        [
            own[own[topic]]
            .sample(
                min(SAMPLE_SIZE.get(topic, DEFAULT_SAMPLE), own[topic].sum()),
                random_state=SEED,
            )
            .assign(topic=topic)
            for topic in topic_columns
            if topic != "football"
        ]
    )
    samples.assign(is_topic_label="")[
        ["topic", "timestamp", "message", "is_topic_label"]
    ].to_csv(LABEL_FILE, index=False)
    print(f"Wrote {len(samples)} rows to {LABEL_FILE}")

## Hand check 2: the words added after hand check 1

Five matched messages per added word (student labels `is_topic_label` y/n).

In [ ]:
ADDED_WORDS = {
    "birthdays": ["jarige?"],
    "bad news": ["verschrikkelijke?", "beterschap"],
    "food": ["bbq", "barbecue"],
    "holiday": ["vliegen", "vliegtuig", "vliegveld"],
    "festive days": ["carnavals?", "koningsdag"],
}
ADDED_LABEL_FILE = PROCESSED / "topic-label-check-added.csv"
ADDED_SAMPLE = 5

if ADDED_LABEL_FILE.exists():
    print(f"{ADDED_LABEL_FILE.name} exists, not overwritten")
else:
    rows = []
    for topic, added in ADDED_WORDS.items():
        for word in added:
            hits = own[is_text & text.str.contains(words(word))]
            rows.append(
                hits.sample(min(ADDED_SAMPLE, len(hits)), random_state=SEED).assign(
                    topic=topic, word=word
                )
            )
    pd.concat(rows).assign(is_topic_label="")[
        ["topic", "word", "timestamp", "message", "is_topic_label"]
    ].to_csv(ADDED_LABEL_FILE, index=False)
    print(f"Wrote {sum(len(r) for r in rows)} rows to {ADDED_LABEL_FILE}")

## Overlap between topics

Overlap (student): of the conversations that mention topic A or topic B, the share that mention both. One number per pair, the same both ways.

In [ ]:
from itertools import combinations

both = per_conversation.T.astype(int) @ per_conversation.astype(
    int
)  # conversations with both topics
size = per_conversation.sum()
overlap = pd.Series(
    {
        (a, b): both.loc[a, b] / (size[a] + size[b] - both.loc[a, b])
        for a, b in combinations(topic_columns, 2)
    },
    name="overlap",
).rename_axis(["topic_a", "topic_b"])

pairs = pd.DataFrame(
    {"overlap": overlap.round(3), "both": [both.loc[a, b] for a, b in overlap.index]}
)
print(
    f"{len(pairs)} pairs; overlap {overlap.min():.3f}–{overlap.max():.3f}, median {overlap.median():.3f}"
)
pairs.sort_values("overlap", ascending=False).head(15)

## The network graph

Dot = topic (size = conversations), line = overlap above the cut-off (student: 0.12), thicker = more overlap. Having a drink sits in the middle (student); the topics with a line to it form the inner ring, the rest the outer ring. A 6-hour gap check gives the Spearman number in the footnote.

In [ ]:
from itertools import permutations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

CUT_OFF = 0.12  # student: more topics in the graph than at 0.15
LEFT_OUT = [
    "politics",
    "festive days",
    "money",
]  # no overlap above the cut-off with any topic
BEHAVIOUR = ["laughter"]  # student: laughter is behaviour, not a topic
FOCUS = "having a drink"  # student: pinned in the middle, it and its lines in the accent colour
ACCENT, BASE = (
    "#f6a65a",
    "#9aa5b1",
)  # light orange (student); the rest a quiet grey-blue
LINE_WIDTH = 20  # points per 1.0 overlap: twice the overlap is twice as thick (student)
DOT_AREA = (
    4  # points² per conversation: dot area grows with the number of conversations
)
TITLE = "Having a drink has the strongest ties with other social topics in the group"  # student

graph_topics = [t for t in topic_columns if t not in LEFT_OUT + BEHAVIOUR]
graph = nx.Graph()
graph.add_nodes_from(graph_topics)
for (a, b), value in overlap.items():
    if a in graph_topics and b in graph_topics:
        graph.add_edge(a, b, weight=value)
shown = overlap[
    [
        a in graph_topics and b in graph_topics and v > CUT_OFF
        for (a, b), v in overlap.items()
    ]
]


def overlaps_at(gap_hours: int) -> pd.Series:
    """Overlap per pair of graph topics when a conversation ends after gap_hours of silence."""
    conversation = (own["timestamp"].diff() > pd.Timedelta(hours=gap_hours)).cumsum()
    topics_in = own.groupby(conversation)[graph_topics].any()
    n_both = topics_in.T.astype(int) @ topics_in.astype(int)
    n = topics_in.sum()
    return pd.Series(
        {
            (a, b): n_both.loc[a, b] / (n[a] + n[b] - n_both.loc[a, b])
            for a, b in combinations(graph_topics, 2)
        }
    )


gap_rho = spearmanr(overlaps_at(GAP_HOURS), overlaps_at(6)).statistic


def closest_pass(positions: dict) -> float:
    """Smallest distance between a shown line and a dot that is not one of its two ends."""
    gaps = []
    for a, b in shown.index:
        p, q = np.array(positions[a]), np.array(positions[b])
        for topic in graph_topics:
            if topic in (a, b):
                continue
            r = np.array(positions[topic])
            t = np.clip(np.dot(r - p, q - p) / np.dot(q - p, q - p), 0, 1)
            gaps.append(np.linalg.norm(r - (p + t * (q - p))))
    return min(gaps)


# Rings (student: having a drink in the middle): the topics with a line to it on the inner ring, the rest
# on the outer ring. Try every order on both rings and keep the one where lines pass furthest from the
# other dots (student: no line may run through a dot).
inner = sorted(
    t for t in graph_topics if (FOCUS, t) in shown.index or (t, FOCUS) in shown.index
)
outer = sorted(t for t in graph_topics if t != FOCUS and t not in inner)


def ring_layout(inner_order: tuple, outer_order: tuple) -> dict:
    """Focus topic at (0, 0), inner ring radius 1, outer ring radius 2.4 (shifted half a step)."""
    pos = {FOCUS: np.zeros(2)}
    for radius, order, shift in [(1, inner_order, 0.0), (2.4, outer_order, 0.5)]:
        for k, topic in enumerate(order):
            angle = 2 * np.pi * (k + shift) / len(order) + np.pi / 2
            pos[topic] = radius * np.array([np.cos(angle), np.sin(angle)])
    return pos


layouts = [ring_layout(i, o) for i in permutations(inner) for o in permutations(outer)]
positions = max(layouts, key=closest_pass)
print(f"Inner ring: {inner}; outer ring: {outer}")
print(
    f"Closest a line passes another dot: {closest_pass(positions):.2f} (outer ring radius 2.4)"
)

fig, ax = plt.subplots(figsize=(11, 9))
for (a, b), value in shown.items():
    (xa, ya), (xb, yb) = positions[a], positions[b]
    ax.plot(
        [xa, xb],
        [ya, yb],
        color=ACCENT if FOCUS in (a, b) else "0.7",
        linewidth=LINE_WIDTH * value,
        solid_capstyle="round",
        zorder=2 if FOCUS in (a, b) else 1,
    )
centre = np.array(positions[FOCUS])
for topic, (x, y) in positions.items():
    ax.scatter(
        x,
        y,
        s=DOT_AREA * size[topic],
        color=ACCENT if topic == FOCUS else BASE,
        edgecolor="white",
        linewidth=2,
        zorder=3,
    )
    # label on the outside, away from the middle; the focus topic's label goes above it
    away = np.array([x, y]) - centre
    away = away / np.linalg.norm(away) if np.linalg.norm(away) else np.array([0.0, 1.0])
    reach = 12 + size[topic] ** 0.5 / 1.2
    ax.annotate(
        f"{topic}\n{size[topic]}",
        (x, y),
        xytext=(reach * away[0], reach * away[1]),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold" if topic == FOCUS else "normal",
        zorder=4,
        ha="left" if away[0] > 0.3 else "right" if away[0] < -0.3 else "center",
        va="bottom" if away[1] > 0.3 else "top" if away[1] < -0.3 else "center",
        bbox={
            "boxstyle": "round,pad=0.15",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.85,
        },
    )

# Annotation (student): work is the bridge to football and family; placed in the empty space on the right
ax.annotate(
    "Work links the social topics\nto football and family",
    positions["work"],
    xytext=(0.56, 0.22),
    textcoords="axes fraction",
    fontsize=10,
    color="0.3",
    ha="left",
    arrowprops={"arrowstyle": "-", "color": "0.5", "linewidth": 0.8, "shrinkB": 12},
)

fig.text(0.02, 0.98, TITLE, fontsize=15, va="top")
ax.legend(
    handles=[
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            color=BASE,
            markersize=12,
            label="Dot = a topic; the number is how many\nconversations mention it",
        ),
        Line2D(
            [],
            [],
            color="0.7",
            linewidth=LINE_WIDTH * 0.2,
            label="Line = two topics that often come up in the\nsame conversation; thicker = more often together",
        ),
    ],
    loc="upper left",
    bbox_to_anchor=(1.02, 0.88),
    frameon=False,
    fontsize=9,
    labelspacing=2.2,
    handlelength=3.5,
    handletextpad=1.8,
    borderpad=1,
    title="How to read",
    title_fontsize=10,
    alignment="left",
)
fig.text(
    0.02,
    0.01,
    f"Conversations: {len(per_conversation):,}, each ending after {GAP_HOURS} hours without a message. "
    f"Lines only above {CUT_OFF:.0%}.\n"
    f"With a 6-hour gap the order of the overlaps stays the same (Spearman {gap_rho:.2f}).",
    fontsize=9,
    color="0.4",
)
ax.axis("off")
plt.tight_layout(rect=(0, 0.05, 1, 0.94))
fig.savefig("topics-network-draft-8.png", dpi=150, bbox_inches="tight")
print(f"{len(shown)} lines; Spearman 12 h vs 6 h: {gap_rho:.2f}")

## Check: does the graph depend on the 12-hour gap?

Same topics and overlap with a 6-hour gap (student). Shorter conversations hold fewer topics, so overlaps are expected to be lower; the question is whether the same pairs stay on top.

In [ ]:
gap_check = pd.DataFrame({"12 h": overlaps_at(12), "6 h": overlaps_at(6)})
print(
    f"Spearman between the gaps: {spearmanr(gap_check['12 h'], gap_check['6 h']).statistic:.2f}"
)
print(
    f"Lines above {CUT_OFF}: 12 h {(gap_check['12 h'] > CUT_OFF).sum()}, 6 h {(gap_check['6 h'] > CUT_OFF).sum()}"
)
gap_check["rank 12 h"] = gap_check["12 h"].rank(ascending=False).astype(int)
gap_check["rank 6 h"] = gap_check["6 h"].rank(ascending=False).astype(int)
gap_check.sort_values("12 h", ascending=False).head(14).round(3)